In [ ]:
# Cell 1 — Install ML Toolkit
!pip install scikit-learn skl2onnx pandas numpy -q
print("✅ Toolkit installed")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.2/317.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 39.9 MB/s eta 0:00:00
✅ Toolkit installed


In [ ]:
# Cell 2 — Build synthetic data matching sensor ingestion structure
import pandas as pd, numpy as np

# Feature: NH=0, SH=1, MDR=2, VR=3
df = pd.DataFrame({
    'road_type': np.random.choice([0,1,2,3], 5000, p=[0.3,0.3,0.25,0.15]),
    'hour': np.random.randint(0,24, 5000),
    'is_rain': np.random.choice([0,1], 5000, p=[0.7,0.3]),
    'speed_limit': np.random.choice([40,60,80,100], 5000),
    'prev_accidents': np.random.poisson(2, 5000)
})

# Synthesize a high-risk label for training (e.g., Night + Rain + Fast highway)
df['high_risk'] = ((df['hour'].isin([22,23,0,1,2,3,4])) & (df['road_type'] <= 1) & (df['is_rain'] == 1)).astype(int)

X = df.drop('high_risk', axis=1)
y = df['high_risk']

print("✅ Data matrix ready. Rows:", len(X))


✅ Data matrix ready. Rows: 5000


In [ ]:
# Cell 3 — Train the GradientBoosting Classifier
from sklearn.ensemble import GradientBoostingClassifier

model = GradientBoostingClassifier(n_estimators=50, max_depth=4)
model.fit(X, y)
print("✅ Risk AI Training finished")


✅ Risk AI Training finished


In [ ]:
# Cell 4 — Package as ONNX and Export
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
from google.colab import files

initial_type = [('features', FloatTensorType([None, 5]))]
onx = convert_sklearn(model, initial_types=initial_type)

with open('risk_model.onnx', 'wb') as f:
    f.write(onx.SerializeToString())

print(f"✅ Executed. Output size: {len(onx.SerializeToString())//1024} KB")
files.download('risk_model.onnx')


✅ Executed. Output size: 21 KB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>